# Omnibus — first insights from `features.parquet`

Quick look at the 3.8M-row stop-event table. **Two caveats baked in below — read them before forking.**

1. **Overlap.** `Daten_Linie_1_2024-09_2025-08` (full year Line 1) overlaps every event window. Filter on `source_window`, never on date alone. Here we drop the Line-1 window upfront so date-based slicing is safe.
2. **Midnight wrap.** Service days exceed 24h, so a small fraction of `delay_arr_s` rows are off by ~±24h (see `docs/DATA_DEFECTS.md` §9). Filter `|delay| < 2h`.

Other gotchas worth knowing: `door_opened` is unreliable on wide files (§10), and Line-1 has stop-identity nulls in two months (§4) — neither matters here since we exclude Line-1.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path("../data/parquet/features.parquet")
df = pl.read_parquet(DATA)
print(f"raw rows: {df.height:,}")
df.head(3)

## Caveat handling — slice + clean

In [ ]:
# Drop the Line-1 full year so each date appears at most once across event windows.
ev = df.filter(pl.col("source_window") != "Daten_Linie_1_2024-09_2025-08")

# Drop midnight-wrap outliers (|delay| > 2h).
clean = ev.filter(pl.col("delay_arr_s").abs() < 7200)

print(f"event-window rows: {ev.height:,}")
print(f"after delay clean: {clean.height:,}  ({ev.height-clean.height} dropped)")

clean.group_by("source_window").len().sort("len", descending=True)

## Insight 1 — Which lines are most *unreliable*?

Reliability = low σ(arrival delay). Riders accept a bus that's always 4 min late; they riot at the one that's sometimes 0, sometimes 12. This is the headline metric for Scene B.

Filter to productive arrivals (real passenger-serving stops), require ≥5k samples to avoid lines with a single bad week skewing the ranking.

In [ ]:
worst = (
    clean
    .filter(pl.col("productive_arr"))
    .group_by("line")
    .agg(
        pl.col("delay_arr_s").std().alias("sigma_s"),
        pl.col("delay_arr_s").median().alias("median_s"),
        pl.col("delay_arr_s").quantile(0.9).alias("p90_s"),
        pl.len().alias("n"),
    )
    .filter(pl.col("n") > 5000)
    .sort("sigma_s", descending=True)
    .head(12)
)
worst

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(worst["line"], worst["sigma_s"], color="#c0392b")
ax.set_xlabel("σ(arrival delay) [s]")
ax.set_title("Top 12 most unreliable lines (n≥5k productive arrivals)")
ax.invert_yaxis()
for i, (s, n) in enumerate(zip(worst["sigma_s"], worst["n"])):
    ax.text(s + 3, i, f"n={n:,}", va="center", fontsize=8, color="#555")
fig.tight_layout()
plt.show()

**Read:** **X4** is the runaway outlier — σ≈290s, almost 2× any other line. It's a commuter express, longer route + freeway segments = more places for variance to compound. Lines 9, 4, 8, 11 form the urban "second tier" worth investigating per-stop.

For the demo: X4 is the natural Scene B lead-in ("here's the worst line in the city — *why?*").

## Insight 2 — The June 2024 flood, day by day

Replay the flood window. Watch the median and p90 climb, then *stay elevated* even after rainfall stops — the recovery half-life is the real story for Scene C.

Peak Donau Meldestufe 4 was reached **2 June 2024** (≈5.5 m gauge).

In [ ]:
flood = clean.filter(pl.col("source_window").str.starts_with("26.05.2024"))
by_day = (
    flood
    .group_by("operating_day")
    .agg(
        pl.col("delay_arr_s").median().alias("median_s"),
        pl.col("delay_arr_s").quantile(0.9).alias("p90_s"),
        pl.col("delay_arr_s").std().alias("sigma_s"),
        pl.col("precip_mm").mean().alias("precip_mean"),
        pl.len().alias("n"),
    )
    .sort("operating_day")
)
by_day

In [ ]:
days = by_day["operating_day"].to_list()
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})

ax1.plot(days, by_day["median_s"], marker="o", color="#2980b9", label="median delay")
ax1.plot(days, by_day["p90_s"], marker="s", color="#c0392b", label="p90 delay")
ax1.axvline(by_day["operating_day"][7], color="#888", linestyle="--", alpha=0.6, label="flood peak (2 Jun)")
ax1.set_ylabel("arrival delay [s]")
ax1.set_title("Delay during the June 2024 flood window")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.bar(days, by_day["precip_mean"], color="#7f8c8d", width=0.7)
ax2.set_ylabel("mean hourly precip [mm]")
ax2.set_xlabel("operating day")
ax2.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()

**Read:** Median delay roughly **doubles** post-peak (33s → 51s) and p90 jumps **+50%** (160 → 240s). And it *stays there* through the rest of the window — the network is still degraded 5 days after the rain stopped. That tail is exactly what Scene C should visualise.

## Insight 3 — Christmas market vs October baseline (by hour of day)

Both windows are October–December 2024, both two-week stretches. Compare median delay across the day to see when the city *actually* slows down.

In [ ]:
def by_hour(d: pl.DataFrame) -> pl.DataFrame:
    return (
        d.with_columns(hour=pl.col("ts_arrival_planned").dt.hour())
         .group_by("hour")
         .agg(pl.col("delay_arr_s").median().alias("median_s"),
              pl.col("delay_arr_s").quantile(0.9).alias("p90_s"))
         .sort("hour")
    )

xmas = by_hour(clean.filter(pl.col("source_window").str.starts_with("15.12.2024")))
base = by_hour(clean.filter(pl.col("source_window") == "06.10.2024_19.10.2024_ITCS"))

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(xmas["hour"], xmas["median_s"], marker="o", label="Christmas market (15–25 Dec 2024)", color="#27ae60")
ax.plot(base["hour"], base["median_s"], marker="o", label="Baseline (6–19 Oct 2024)", color="#7f8c8d")
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("hour of day")
ax.set_ylabel("median arrival delay [s]")
ax.set_title("Median delay by hour — Christmas market vs October baseline")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

**Read (counter-intuitive):** the Christmas market window is **less delayed** at peak commute hours than the October baseline — likely because the school-holiday + party-leisure mix collapses the morning peak, even as evening market traffic shows up. This is why you can't trust a single median; the *variance* and *route-specific* views still tell a different story. Worth a deeper cut per line.

## Insight 4 — Does rain actually hurt buses?

Bucket every stop-event by hourly precip and check median + σ. Quick sanity for Scene B's "weather-coupled" variance class.

In [ ]:
buckets = (
    clean
    .with_columns(
        precip_bin=pl.when(pl.col("precip_mm") < 0.1).then(pl.lit("0  (dry)"))
                    .when(pl.col("precip_mm") < 1.0).then(pl.lit("1  (drizzle <1mm)"))
                    .when(pl.col("precip_mm") < 3.0).then(pl.lit("2  (light 1–3mm)"))
                    .when(pl.col("precip_mm") < 7.0).then(pl.lit("3  (moderate 3–7mm)"))
                    .otherwise(pl.lit("4  (heavy ≥7mm)"))
    )
    .group_by("precip_bin")
    .agg(
        pl.col("delay_arr_s").median().alias("median_s"),
        pl.col("delay_arr_s").std().alias("sigma_s"),
        pl.len().alias("n"),
    )
    .sort("precip_bin")
)
buckets

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(buckets))
ax.bar([i - 0.2 for i in x], buckets["median_s"], width=0.4, label="median", color="#2980b9")
ax.bar([i + 0.2 for i in x], buckets["sigma_s"], width=0.4, label="σ", color="#c0392b")
ax.set_xticks(list(x))
ax.set_xticklabels(buckets["precip_bin"], rotation=15, ha="right")
ax.set_ylabel("seconds")
ax.set_title("Arrival delay by hourly precip bucket")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

**Read:** σ climbs with precip — rain doesn't just delay buses on average, it makes them *unpredictable*. Heavy-rain hours look very different from dry hours, which validates "weather-coupled variance" as a real Scene B label.